# Entrenamiento continuo CORTEX-LM en Colab

Este cuaderno descarga o ejecuta el entrenamiento por lotes. La descarga y el entrenamiento son acciones separadas: la Celda 3 solo descarga; la Celda 4 entrena.

## Preparación segura
1. En Colab, guarda un token nuevo en **Secrets** con el nombre `HF_TOKEN` y habilita el acceso para este cuaderno. No lo pegues en una celda ni lo compartas.
2. Selecciona un runtime CPU con RAM suficiente; el motor de entrenamiento es C++/OpenMP y no usa la GPU T4.
3. Este cuaderno usa por defecto la rama de trabajo actual. Puedes fijar otra con la variable `CORTEX_BRANCH`.


In [ ]:
# Celda 1: leer la credencial desde Colab Secrets sin imprimirla
import os
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Configura HF_TOKEN en Colab Secrets; no lo pegues en el cuaderno.')
os.environ['HF_TOKEN'] = hf_token
del hf_token
!pip install -q huggingface_hub datasets
print('Dependencias y secreto configurados de forma privada.')


In [ ]:
# Celda 2: clonar/actualizar la rama solicitada y compilar
import os, subprocess
from pathlib import Path
BRANCH = os.environ.get('CORTEX_BRANCH', 'arena/b0c825e4-nuby')
REPO = Path('/content/nuby_repo')
URL = 'https://github.com/alexsndersoto04-source/nuby-.git'
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '-B', BRANCH, 'FETCH_HEAD'], check=True)
%cd /content/nuby_repo
!make -C cortex-lm portable -j2


In [ ]:
# Celda 3: descargar solamente; --solo-descargar ya está soportado por el script
!python cloud/entrenar_wikipedia_continuo.py --solo-descargar
!ls -lh cortex-lm/data/cerebro_macro_1m.bin cortex-lm/data/registro_wikipedia.json


In [ ]:
# Celda 4: iniciar entrenamiento por lotes (hasta el límite configurado del script)
!python cloud/entrenar_wikipedia_continuo.py --max-lotes 1

print('\n--- Registro actual ---')
!cat cortex-lm/data/registro_wikipedia.json


---
## Para continuar en otra sesión
1. Abre este cuaderno y confirma que el secreto `HF_TOKEN` está habilitado en Colab Secrets.
2. Ejecuta las celdas en orden: la Celda 3 solo descarga; la Celda 4 sí entrena.
3. No pegues credenciales en celdas ni en el chat.
